# Shared detector: potholes + road signs — Kaggle training

Before running:
1. **Settings → Accelerator → GPU T4 x2**, **Settings → Internet → On**
2. **Add Input → Datasets → `rdd-2022`** (aliabdelmenam)
3. **Add Input → your sign dataset.** Easiest: export the Roboflow dataset as *YOLOv8* zip, then on Kaggle **Create → New Dataset**, upload the zip (keep it private), and attach it here. Give it a name containing `sign`.

Outputs: weights, TFLite exports, and `summary_shared_v1.json` with pothole, India, and **clear vs occluded sign** results.

Datasets: RDD2022 (Arya et al., CC BY-SA 4.0); Indian traffic signs (Roboflow, Tech Mahindra Altavec, CC BY 4.0). Attribution required.


In [ ]:
MODEL = "yolo11n.pt"      # check against the current Ultralytics docs
IMGSZ = 640
EPOCHS = 60
TIME_HOURS = 9.5
BATCH = 32
RUN_NAME = "shared_v1"
WORK = "/kaggle/working"
REPO = "https://github.com/divyanshkulshrestha23/road-anomaly-detection"
RDD_OUT = f"{WORK}/rdd_yolo"
SIGN_OUT = f"{WORK}/signs_yolo"


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import glob, os
dirs = sorted({p.rstrip("/") for p in glob.glob("/kaggle/input/**/", recursive=True)}, key=len)
RAW_RDD = next(p for p in dirs if "rdd" in p.lower())
SIGN_SRC = next((p for p in dirs if "sign" in p.lower() and glob.glob(p + "/**/data.yaml", recursive=True)), None)
assert SIGN_SRC, "Sign dataset not found: attach it and make sure its name contains 'sign'"
print("RDD:", RAW_RDD); print("Signs:", SIGN_SRC)


In [ ]:
if not os.path.exists(f"{WORK}/road-anomaly-detection"):
    !git clone -q {REPO} {WORK}/road-anomaly-detection
else:
    !git -C {WORK}/road-anomaly-detection pull -q
%cd {WORK}/road-anomaly-detection
!pip install -q -r detector/requirements.txt
!git log --oneline -1


## Check sign classes
Every source class should map to a sign group or `None` (dropped on purpose). `UNMAPPED` means `detector/configs/signs_map.yaml` needs updating.

In [ ]:
!python detector/signs_prepare.py --src {SIGN_SRC} --inspect

## Prepare both datasets

In [ ]:
!python detector/rdd_prepare.py --src {RAW_RDD} --out {RDD_OUT}
!python detector/signs_prepare.py --src {SIGN_SRC} --out {SIGN_OUT} --rdd {RDD_OUT}
assert os.path.exists(f"{SIGN_OUT}/data_combined.yaml")


## Train on potholes + signs

In [ ]:
from ultralytics import YOLO
model = YOLO(MODEL)
model.train(data=f"{SIGN_OUT}/data_combined.yaml", epochs=EPOCHS, time=TIME_HOURS, imgsz=IMGSZ,
            batch=BATCH, project=f"{WORK}/runs", name=RUN_NAME, exist_ok=True,
            patience=15, seed=42, device=0, workers=4)


## Evaluate: potholes (all, India) and signs (clear, 25% and 50% occluded)

In [ ]:
import glob
from ultralytics import YOLO
best = f"{WORK}/runs/{RUN_NAME}/weights/best.pt"
model = YOLO(best)
evals = [("pothole_all", f"{RDD_OUT}/data.yaml"), ("pothole_india", f"{RDD_OUT}/data_india.yaml"),
         ("signs_clear", f"{SIGN_OUT}/data_signs.yaml")]
evals += [(os.path.basename(p)[len("data_signs_"):-5], p) for p in sorted(glob.glob(f"{SIGN_OUT}/data_signs_*occ*.yaml"))]
results = {}
for tag, data in evals:
    m = model.val(data=data, split="test", imgsz=IMGSZ, batch=BATCH, plots=False, verbose=False)
    present = [int(i) for i in m.box.ap_class_index]
    results[tag] = {"mAP50": round(float(m.box.map50), 4), "mAP50_95": round(float(m.box.map), 4),
                    "precision": round(float(m.box.mp), 4), "recall": round(float(m.box.mr), 4),
                    "per_class_mAP50_95": {model.names[i]: round(float(m.box.maps[i]), 4) for i in present}}
    print(tag, results[tag])


In [ ]:
import os, shutil
from ultralytics import YOLO
export_dir = f"{WORK}/exports"; os.makedirs(export_dir, exist_ok=True)
for kw in [dict(half=True), dict(int8=True, data=f"{SIGN_OUT}/data_combined.yaml")]:
    try:
        YOLO(best).export(format="tflite", imgsz=IMGSZ, **kw)
    except Exception as e:
        print("Export failed for", kw, "->", e)
for f in glob.glob(f"{WORK}/runs/{RUN_NAME}/weights/**/*.tflite", recursive=True):
    shutil.copy2(f, export_dir)
with open(f"{export_dir}/labels.txt", "w") as f:
    f.write("\n".join(model.names[i] for i in sorted(model.names)) + "\n")
print(os.listdir(export_dir))


In [ ]:
import json, datetime, subprocess, torch
summary = {"run": RUN_NAME, "model": MODEL, "imgsz": IMGSZ, "epochs": EPOCHS, "time_cap_h": TIME_HOURS,
           "batch": BATCH, "platform": "kaggle", "date": datetime.datetime.now().isoformat(timespec="seconds"),
           "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
           "repo_commit": subprocess.check_output(["git", "rev-parse", "--short", "HEAD"]).decode().strip(),
           "classes": model.names, "results": results, "exports": sorted(os.listdir(export_dir))}
with open(f"{WORK}/summary_{RUN_NAME}.json", "w") as f:
    json.dump(summary, f, indent=2, default=str)
for fn in [f"{RDD_OUT}/prepare_summary.json", f"{SIGN_OUT}/signs_summary.json"]:
    shutil.copy2(fn, WORK)
shutil.rmtree(RDD_OUT, ignore_errors=True); shutil.rmtree(SIGN_OUT, ignore_errors=True)
shutil.rmtree(f"{WORK}/road-anomaly-detection", ignore_errors=True)
print(json.dumps(summary["results"], indent=2))
